In [32]:
! pip install agent-framework --pre


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
!pip show agent-framework

Name: agent-framework
Version: 1.0.0b251223
Summary: Microsoft Agent Framework for building AI Agents with Python. This package contains all the core and optional packages.
Home-page: 
Author: 
Author-email: Microsoft <af-support@microsoft.com>
License: 
Location: C:\Users\yethish.poojarira\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: agent-framework-core
Required-by: 


In [30]:
!python -m pip uninstall -y agent-framework

Found existing installation: agent-framework 1.0.0b251223
Uninstalling agent-framework-1.0.0b251223:
  Successfully uninstalled agent-framework-1.0.0b251223


In [25]:
!python -m pip install agent-framework==1.0.0b251223

     -------------------------------------- 359.8/359.8 kB 1.5 MB/s eta 0:00:00
  Attempting uninstall: agent-framework-core
    Found existing installation: agent-framework-core 1.0.0b260116
    Uninstalling agent-framework-core-1.0.0b260116:
      Successfully uninstalled agent-framework-core-1.0.0b260116



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [162]:
import asyncio
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential
from agent_framework import ai_function
from typing import Annotated
from pydantic import Field

know_info = {
    "company_name" : "None",
    "role" : "None"
}

@ai_function(name="update_company_name", description="Update the known company name from the user's message")
def update_company_name(company: Annotated[str, Field(description="The Company Name User Is Preparing For.")] = None) -> str:
    """Update the Company Name for which the user is preparing for."""
    print(f"Tool called: update_company_name with {company}")
    know_info["company_name"] = company
    return f"Updated known info: company={know_info['company_name']}, role={know_info['role']}"

@ai_function(name="update_job_role", description="Update the known job role from the user's message")
def update_job_role(role: Annotated[str, Field(description="The Job Role User Is Preparing For.")] = None) -> str:
    """Update the Job Role for which the user is preparing for."""
    print(f"Tool called: update_job_role with {role}")
    if role:
        know_info["role"] = role
    return f"Updated known info: company={know_info['company_name']}, role={know_info['role']}"

agent = AzureOpenAIChatClient(credential=AzureCliCredential()).as_agent(
    instructions='''
    - Extract any company name mentioned in the message using the update_company_name tool.
    - Extract any job role mentioned in the message using the update_job_role tool.
    - Even if only one of company name or job role is mentioned, use the respective tool to update the known info.
    - If the user specifically asks about interview preparation or both company and role are known, then provide career guidance as follows:
      - Based on the given company name and job role, provide key topics to cover for a fresher to crack the interview round.
      - Perform web-based research to provide accurate and up-to-date information.
      - Consider social media platforms like LinkedIn, Glassdoor, and others for insights on interview experiences.
      - Also consider official job descriptions and company career pages.
      - Consider the latest trends and technologies related to the job role.
      - Provide concise and relevant information.
      - Consider the list of info to be known for a fresher.
      - Don't include greetings or sign-offs.
      - If DSA/Database is needed, also mention the topics to be covered in DSA.
      - Return the response in bullet point format.
      - At the end, provide the sources of resources used for research based on which you were able to generate the response.
      - Also include the links of resources for further reading.
      - Both company and job role should be provided by the user before giving tips; ask for missing information if needed.
      - Analyze the conversation to understand if the info is missing or not.
    - Otherwise, respond naturally to the user.
    ''',
    tools=[update_company_name, update_job_role],
    name="Joker"
)

In [167]:
from agent_framework import ChatMessage

chat_history = [    
    ChatMessage(
        role="system", 
        text='''
        Use this as the conversation context.
        Prefer recent messages over older messages.
        Keep track of known information about the user:
        - Company Name: None
        - Job Role: None
        Update this information when the user provides it.
        '''
        )
]


def add_message(role: str, content: str)-> list[ChatMessage]:
    """Add a message to the chat history."""
    global chat_history
    chat_history.append(ChatMessage(role=role, text=content))


def print_history():
    """Print the chat history."""
    global chat_history
    for msg in chat_history:
        print(f"{msg.role}: {msg.text}")
    print("----- End of chat history -----")


while True:
    user_input = input("👤: ")
    add_message("user", user_input)
    if user_input.lower() in ["exit", "quit"]:
        break
    result = await agent.run(chat_history)
    add_message("assistant", result.text)
    print_history()
    print("Final known info:", know_info)


Tool called: update_company_name with Microsoft
system: 
        Use this as the conversation context.
        Prefer recent messages over older messages.
        Keep track of known information about the user:
        - Company Name: None
        - Job Role: None
        Update this information when the user provides it.
        
user: I'm preparing for Microsoft
assistant: What job role are you preparing for at Microsoft?
----- End of chat history -----
Final known info: {'company_name': 'Microsoft', 'role': 'None'}
Tool called: update_company_name with IBM
system: 
        Use this as the conversation context.
        Prefer recent messages over older messages.
        Keep track of known information about the user:
        - Company Name: None
        - Job Role: None
        Update this information when the user provides it.
        
user: I'm preparing for Microsoft
assistant: What job role are you preparing for at Microsoft?
user: I want to prepare for IBM
assistant: What job ro

In [166]:
from agent_framework import ChatMessage, TextContent
import os

def clrscr():
    """Clear the screen."""
    os.system('cls' if os.name == 'nt' else 'clear')

# Create a message with text
user_msg = ChatMessage(role="user", text="What's the weather?")
print(user_msg.text)  # "What's the weather?"

# Create a message with role string
system_msg = ChatMessage(role="system", text="You are a helpful assistant.")

# Create a message with contents
assistant_msg = ChatMessage(
    role="assistant",
    contents=[TextContent(text="The weather is sunny!")],
)
print(assistant_msg.text)  # "The weather is sunny!"


What's the weather?
The weather is sunny!


In [160]:
# add_message("user", "I'm preparing for Microsoft")
# result = await agent.run(chat_history)
# add_message("assistant", result.text)
# print_history()
# print("Final known info:", know_info)

# add_message("user", "I want to prepare for IBM")
# result = await agent.run(chat_history)
# add_message("assistant", result.text)
# print_history()
# print("Final known info:", know_info)

# add_message("user", "for SE role")
# result = await agent.run(chat_history)
# add_message("assistant", result.text)
# print_history()
# print("Final known info:", know_info)

Tool called: update_company_name with IBM
Tool called: update_job_role with SE
system: 
        Use this as the conversation context.
        Prefer recent messages over older messages.
        Keep track of known information about the user:
        - Company Name: None
        - Job Role: None
        Update this information when the user provides it.
        
user: I'm preparing for Microsoft
assistant: What job role are you preparing for at Microsoft?
user: I want to prepare for IBM
assistant: What job role are you preparing for at IBM?
user: for SE role
assistant: For preparing for a Software Engineer (SE) role at IBM, here are some key topics to cover:

- **Technical Skills:**
  - **Programming Languages:** Proficiency in Java, Python, or C++.
  - **Data Structures and Algorithms (DSA):**
    - Arrays, Linked Lists, Trees, Graphs, Stacks, Queues.
    - Sorting and Searching algorithms.
    - Dynamic Programming concepts.
  - **System Design:** Basics of system architecture, distri